# Days 8-9: Two-tower model
## Framework decision: Keras
I'm using **Keras**, not PyTorch. I already know Keras, and the two-tower model is the hardest part of this project, so I'd rather learn one hard thing at a time: the model, not a new framework as well.

**What I give up:** on Day 9 the in-batch softmax loss needs a small custom piece in Keras. Standard `model.fit` losses expect one label per row, but in-batch loss compares every customer in the batch against every article in the batch. That's fine, I just know it's coming.

## Step 3: Dot product vs cosine (on paper)
- Customer **u = [3, 0, 4]**
- Article **a = [1, 0, 0]**
- Article **b = [6, 0, 8]** (same direction as u, twice as long)

**1. Lengths** (square root of the sum of squares)
- |u| = √(9 + 0 + 16) = √25 = **5**
- |a| = √(1 + 0 + 0) = √1 = **1**
- |b| = √(36 + 0 + 64) = √100 = **10**

**2. Dot products** (multiply matching positions, add)
- u·a = 3×1 + 0×0 + 4×0 = **3**
- u·b = 3×6 + 0×0 + 4×8 = 18 + 32 = **50**

**3. Cosines** = dot ÷ (length × length)
- cos(u, a) = 3 ÷ (5 × 1) = **0.6**
- cos(u, b) = 50 ÷ (5 × 10) = **1.0**, the highest possible: same direction exactly

**What it shows:** with the plain dot product, b scores 50 partly because it's **long**, not only because it points the right way. Make b twice as long again and its dot product doubles to 100, but its cosine stays 1.0. Popular articles tend to get longer vectors, so the plain dot product quietly pushes popular items up: the Day 2 popularity bias again.

**Why normalise the tower outputs?** Dividing each vector by its length (L2 normalisation) makes every vector length 1, so the dot product becomes the cosine and the model scores by **direction** (taste) only, not by vector size (popularity).

**Day 9 preview:** cosine only ranges from −1 to 1, too narrow for the softmax loss, so we divide by a small **temperature**.

In [1]:
# Check the paper numbers with numpy
import numpy as np

u = np.array([3, 0, 4])
a = np.array([1, 0, 0])
b = np.array([6, 0, 8])

print('Lengths:', np.linalg.norm(u), np.linalg.norm(a), np.linalg.norm(b), '| expected 5 1 10')
print('Dot products:', u @ a, u @ b, '| expected 3 50')
print('Cosines:', (u @ a) / (np.linalg.norm(u) * np.linalg.norm(a)),
      (u @ b) / (np.linalg.norm(u) * np.linalg.norm(b)), '| expected 0.6 1.0')

# After L2 normalisation, the dot product IS the cosine
u_unit = u / np.linalg.norm(u)
b_unit = b / np.linalg.norm(b)
print('Normalised dot product u·b:', u_unit @ b_unit, '| expected 1.0 (= the cosine)')

Lengths: 5.0 1.0 10.0 | expected 5 1 10
Dot products: 3 50 | expected 3 50
Cosines: 0.6 1.0 | expected 0.6 1.0
Normalised dot product u·b: 1.0 | expected 1.0 (= the cosine)


## Step 4: Load the train table and the Day 5 features
The network only ever sees integers from these mappings. If one loads wrong (remember the JSON trap: number keys turned into text), training runs without errors and just gives bad numbers on Day 9. So every size is checked against Day 5.

In [2]:
import json
import pandas as pd

FEATURES_FOLDER = '../data/processed/features'

train = pd.read_parquet('../data/processed/train.parquet')
print('Train rows:', len(train), '| expected 2,638,103')

Train rows: 2638103 | expected 2,638,103


In [3]:
def load_vocab(name):
    """Day 5 mapping as a dict: value -> number (1..N, 0 = unknown)."""
    table = pd.read_parquet(FEATURES_FOLDER + '/vocab_' + name + '.parquet')
    return dict(zip(table['value'], table['index']))

customer_vocab = load_vocab('customer')
article_vocab = load_vocab('article')
product_type_vocab = load_vocab('product_type_no')
department_vocab = load_vocab('department_no')
index_group_vocab = load_vocab('index_group_name')
age_bucket_vocab = load_vocab('age_bucket')

print('Customers:    ', len(customer_vocab), '| expected 444,431')
print('Articles:     ', len(article_vocab), '| expected 38,603')
print('Product types:', len(product_type_vocab), '| expected 123')
print('Departments:  ', len(department_vocab), '| expected 286')
print('Index groups: ', len(index_group_vocab), '| expected 5')
print('Age buckets:  ', len(age_bucket_vocab), '| expected 7')

Customers:     444431 | expected 444,431
Articles:      38603 | expected 38,603
Product types: 123 | expected 123
Departments:   286 | expected 286
Index groups:  5 | expected 5
Age buckets:   7 | expected 7


In [4]:
# The JSON trap check: keys must still be numbers, not text
first_customer = list(customer_vocab.keys())[0]
first_article = list(article_vocab.keys())[0]
print('Customer key:', first_customer, type(first_customer), '| expected a number, not str')
print('Article key: ', first_article, type(first_article), '| expected a number, not str')

# Numbers must run 1..N with no gaps (0 is kept for unknown)
print('Customer numbers:', min(customer_vocab.values()), 'to', max(customer_vocab.values()), '| expected 1 to 444431')
print('Article numbers: ', min(article_vocab.values()), 'to', max(article_vocab.values()), '| expected 1 to 38603')

Customer key: 0 <class 'int'> | expected a number, not str
Article key:  108775015 <class 'int'> | expected a number, not str
Customer numbers: 1 to 444431 | expected 1 to 444431
Article numbers:  1 to 38603 | expected 1 to 38603


In [5]:
# Age bucket edges and the overall median price (saved together in settings.json on Day 5)
with open(FEATURES_FOLDER + '/settings.json') as f:
    feature_settings = json.load(f)

age_edges = feature_settings['age_edges']
overall_median_price = feature_settings['overall_median_price']
print('Age edges:', age_edges, '| expected [16, 20, 25, 30, 40, 50, 60, 100]')
print('Buckets from the edges:', len(age_edges) - 1, '| expected 7')
print('Overall median price:', round(overall_median_price, 4), '| expected 0.0212')

Age edges: [16, 20, 25, 30, 40, 50, 60, 100] | expected [16, 20, 25, 30, 40, 50, 60, 100]
Buckets from the edges: 7 | expected 7
Overall median price: 0.0212 | expected 0.0212


In [6]:
# Median price per article, and the product-type fallback for articles without one
article_price = pd.read_parquet(FEATURES_FOLDER + '/article_median_price.parquet')
type_price = pd.read_parquet(FEATURES_FOLDER + '/type_median_price.parquet')

print('Article prices:', len(article_price), '| expected 38,603 (one per article)')
print('Type prices:   ', len(type_price), '| expected 123 (one per product type)')
print('Missing prices:', article_price['median_price'].isna().sum(), type_price['median_price'].isna().sum(), '| expected 0 0')

Article prices: 38603 | expected 38,603 (one per article)
Type prices:    123 | expected 123 (one per product type)
Missing prices: 0 0 | expected 0 0


In [7]:
#Building The Training Pairs
customers = pd.read_parquet('../data/RAW/customers.parquet')
customers_idx = pd.read_parquet('../data/processed/customer_join_idx.parquet')

In [8]:
customer_table = customers.merge(
    customers_idx,
    on='customer_id',
    how='left',
    validate='one_to_one',    # each customer_id once on both sides, or pandas raises an error
)

In [9]:
customer_table = customer_table[
    customer_table['customer_join_idx'].isin(train['customer_join_idx'].unique())
].copy()

In [10]:
print(len(customer_table))

444431


In [11]:
# Vocabularies for the customer columns (Step 4 only loaded age_bucket_vocab)
fashion_news_frequency_vocab = load_vocab('fashion_news_frequency')
club_member_status_vocab = load_vocab('club_member_status')
print('News frequency:', fashion_news_frequency_vocab)
print('Club status:   ', club_member_status_vocab)

News frequency: {'Monthly': 1, 'NONE': 2, 'Regularly': 3}
Club status:    {'ACTIVE': 1, 'LEFT CLUB': 2, 'PRE-CREATE': 3}


In [12]:
#Convert FN AND Active \
customer_table['FN']=customer_table['FN'].fillna(0)
customer_table['Active'] = customer_table['Active'].fillna(0)


In [13]:
customer_table['fashion_news_frequency'] = customer_table['fashion_news_frequency'].replace('None', 'NONE')

In [14]:
customer_table['fashion_news_frequency'] = (
    customer_table['fashion_news_frequency']
    .map(fashion_news_frequency_vocab)
    .fillna(0)
)

In [15]:
customer_table['club_member_status'] = (
    customer_table['club_member_status']
    .map(club_member_status_vocab)
    .fillna(0)
)

In [16]:
# age_edges was loaded from settings.json in Step 4: use the saved copy, don't retype it
customer_table['age'] = pd.cut(
    customer_table['age'],
    bins=age_edges,
    right=False,
    labels=age_edges[:-1]
).astype('float')

In [17]:
customer_table['age'] = (
    customer_table['age']
    .map(age_bucket_vocab)
    .fillna(0)
)

In [18]:
customer_table['customer_idx'] = (
    customer_table['customer_join_idx']
    .map(customer_vocab)
)

In [19]:
# These columns hold vocabulary NUMBERS, not raw values: the name should say so
# (otherwise someone at serving time passes age=34 into an embedding with 8 rows)
customer_table = customer_table.rename(columns={
    'age': 'age_bucket_idx',
    'club_member_status': 'club_member_status_idx',
    'fashion_news_frequency': 'fashion_news_frequency_idx',
})

In [20]:
customer_table = customer_table[
    [
        'customer_join_idx',
        'customer_idx',
        'age_bucket_idx',
        'club_member_status_idx',
        'fashion_news_frequency_idx',
        'FN',
        'Active'
    ]
]
print(len(customer_table), '| expected 444431')
print(customer_table.isna().sum())
print(customer_table.head())

444431 | expected 444431
customer_join_idx             0
customer_idx                  0
age_bucket_idx                0
club_member_status_idx        0
fashion_news_frequency_idx    0
FN                            0
Active                        0
dtype: int64
    customer_join_idx  customer_idx  age_bucket_idx  club_member_status_idx  \
0                   0             1             5.0                     1.0   
1                   1             2             3.0                     1.0   
4                   4             3             6.0                     1.0   
10                 10             4             3.0                     1.0   
13                 13             5             6.0                     1.0   

    fashion_news_frequency_idx   FN  Active  
0                          2.0  0.0     0.0  
1                          2.0  0.0     0.0  
4                          3.0  1.0     1.0  
10                         2.0  0.0     0.0  
13                         3.0  1

In [21]:
articles = pd.read_parquet('../data/RAW/articles.parquet')
article_table = articles[
    articles['article_id'].isin(train['article_id'].unique())
].copy()
print(len(article_table), '| expected 38603')

38603 | expected 38603


In [22]:
article_table['article_idx'] = (
    article_table['article_id'].map(article_vocab)
)

In [23]:
article_columns = [
    'product_code',
    'product_type_no',
    'product_group_name',
    'graphical_appearance_name',
    'colour_group_name',
    'perceived_colour_master_name',
    'department_no',
    'index_group_name',
    'section_name',
    'garment_group_name'
]
for name in article_columns:
    vocab = load_vocab(name)

    article_table[name + '_idx'] = (
        article_table[name]
        .map(vocab)
        .fillna(0)
    )

In [24]:
article_table = article_table.merge(
    article_price,
    on='article_id',
    how='left',
    validate='one_to_one'
)

In [25]:
# Prices are tiny (0.0008 to 0.51), so log1p(x) ≈ x and did nothing (max was still 17.8).
# Plain log squashes ratios. log(0) = -inf, so first check every price is above 0.
print('Smallest median price:', article_table['median_price'].min(), '| expected above 0 (about 0.0008)')
article_table['price_log'] = np.log(article_table['median_price'])

Smallest median price: 0.0007966102 | expected above 0 (about 0.0008)


In [26]:
# Mean and std across the 38,603 ARTICLES (not 2.6M rows, so bestsellers don't drag the mean)
price_log_mean = article_table['price_log'].mean()
price_log_std = article_table['price_log'].std()
article_table['price_scaled'] = (article_table['price_log'] - price_log_mean) / price_log_std

print('Log mean:', round(price_log_mean, 4), '| expected about -3.9165')
print('Log std: ', round(price_log_std, 4), '| expected about 0.7523')
print('Scaled mean / std:', round(article_table['price_scaled'].mean(), 4), '/', round(article_table['price_scaled'].std(), 4), '| expected about 0 / 1')
print('Scaled min / max:', round(article_table['price_scaled'].min(), 2), '/', round(article_table['price_scaled'].max(), 2), '| expected about -4.28 / 4.30 (was 17.8 with log1p)')

Log mean: -3.9165 | expected about -3.9165
Log std:  0.7523 | expected about 0.7523
Scaled mean / std: -0.0 / 1.0 | expected about 0 / 1
Scaled min / max: -4.28 / 4.3 | expected about -4.28 / 4.30 (was 17.8 with log1p)


In [27]:
import json
import os
os.makedirs('../data/processed/features', exist_ok=True)
with open('../data/processed/features/price_scaling.json', 'w') as f:
    json.dump({
        'price_log_mean': float(price_log_mean),
        'price_log_std': float(price_log_std),
        'transform': 'log'   # serving must use np.log, not log1p
    }, f, indent=4)

In [28]:
article_table = article_table[
    [
        'article_id',
        'article_idx'
    ] +
    [name + '_idx' for name in article_columns] +
    ['price_scaled']
].copy()
print(len(article_table), '| expected 38603')
print('Missing values:', article_table.isna().sum().sum(), '| expected 0')
article_table.head()

38603 | expected 38603
Missing values: 0 | expected 0


,article_id,article_idx,product_code_idx,product_type_no_idx,product_group_name_idx,graphical_appearance_name_idx,colour_group_name_idx,perceived_colour_master_name_idx,department_no_idx,index_group_name_idx,section_name_idx,garment_group_name_idx,price_scaled
0,108775015,1,1,48,8,26,2,2,47,3,44,6,-2.009749
1,108775044,2,1,48,8,26,48,17,47,3,44,6,-1.138064
2,110065001,3,2,89,16,26,2,2,13,3,47,19,-1.503698
3,110065002,4,2,89,16,26,48,17,13,3,47,19,-1.575760
4,110065011,5,2,89,16,26,20,1,13,3,47,19,-1.011132


In [29]:
# 5e: merge both tables into the purchases
pairs = train[['customer_join_idx', 'article_id']]
pairs = pairs.merge(customer_table, on='customer_join_idx', how='left', validate='many_to_one')
print(len(pairs), '| expected 2,638,103')
pairs = pairs.merge(article_table, on='article_id', how='left', validate='many_to_one')
print(len(pairs), '| expected 2,638,103')


2638103 | expected 2,638,103


2638103 | expected 2,638,103


In [30]:
pairs = pairs.drop(
    columns=['customer_join_idx', 'article_id']
)
for name in pairs.columns:

    if name in ['FN', 'Active', 'price_scaled']:
        pairs[name] = pairs[name].astype('float32')

    else:
        pairs[name] = pairs[name].astype('int32')


In [31]:
print('Rows:', len(pairs), '| expected 2638103')

print('Missing values:', pairs.isna().sum().sum(), '| expected 0')

print('Smallest customer index:', pairs['customer_idx'].min(), '| expected 1')

print('Smallest article index:', pairs['article_idx'].min(), '| expected 1')

print(pairs.max())

Rows: 2638103 | expected 2638103


Missing values: 0 | expected 0
Smallest customer index: 1 | expected 1
Smallest article index: 1 | expected 1
customer_idx                        444431.000000
age_bucket_idx                           7.000000
club_member_status_idx                   3.000000
fashion_news_frequency_idx               3.000000
FN                                       1.000000
Active                                   1.000000
article_idx                          38603.000000
product_code_idx                     19354.000000
product_type_no_idx                    123.000000
product_group_name_idx                  18.000000
graphical_appearance_name_idx           30.000000
colour_group_name_idx                   50.000000
perceived_colour_master_name_idx        20.000000
department_no_idx                      286.000000
index_group_name_idx                     5.000000
section_name_idx                        56.000000
garment_group_name_idx                  21.000000
price_scaled                            

In [32]:
import tensorflow as tf


In [33]:
pairs =pairs.sample(
    frac=1,
    random_state=42,
).reset_index(drop=True)

In [34]:
features ={}
for name in pairs.columns:
    features[name] =pairs[name].to_numpy()

In [35]:
dataset = tf.data.Dataset.from_tensor_slices(features)

In [36]:
dataset = dataset.shuffle(
    100000,
    seed=42
)
dataset = dataset.batch(
    1024,
    drop_remainder=True
)

In [37]:
dataset = dataset.prefetch(tf.data.AUTOTUNE)

In [38]:
for batch in dataset.take(1):

    for name in batch:
        print(
            name,
            batch[name].shape,
            batch[name].dtype
        )

customer_idx (1024,) <dtype: 'int32'>
age_bucket_idx (1024,) <dtype: 'int32'>
club_member_status_idx (1024,) <dtype: 'int32'>
fashion_news_frequency_idx (1024,) <dtype: 'int32'>
FN (1024,) <dtype: 'float32'>
Active (1024,) <dtype: 'float32'>
article_idx (1024,) <dtype: 'int32'>
product_code_idx (1024,) <dtype: 'int32'>
product_type_no_idx (1024,) <dtype: 'int32'>
product_group_name_idx (1024,) <dtype: 'int32'>
graphical_appearance_name_idx (1024,) <dtype: 'int32'>
colour_group_name_idx (1024,) <dtype: 'int32'>
perceived_colour_master_name_idx (1024,) <dtype: 'int32'>
department_no_idx (1024,) <dtype: 'int32'>
index_group_name_idx (1024,) <dtype: 'int32'>
section_name_idx (1024,) <dtype: 'int32'>
garment_group_name_idx (1024,) <dtype: 'int32'>
price_scaled (1024,) <dtype: 'float32'>


In [39]:
from tensorflow import keras
from keras import layers


In [40]:
# Category features: name -> (input_dim = vocab size + 1 for row 0 = unknown, output_dim = embedding size)
user_categorical = {
    'customer_idx':               (len(customer_vocab) + 1, 32),
    'age_bucket_idx':             (len(age_bucket_vocab) + 1, 4),
    'club_member_status_idx':     (len(club_member_status_vocab) + 1, 4),
    'fashion_news_frequency_idx': (len(fashion_news_frequency_vocab) + 1, 4),
}

# Plain number features (0 or 1): no embedding
user_numeric = ['FN', 'Active']

for name in user_categorical:
    print(name, user_categorical[name])


customer_idx (444432, 32)
age_bucket_idx (8, 4)
club_member_status_idx (4, 4)
fashion_news_frequency_idx (4, 4)


In [41]:
def build_tower(categorical, numeric, tower_name):
    """Features in -> one length-1 vector of 64 numbers out.
    categorical: {name: (input_dim, output_dim)}   numeric: [names of plain number columns]"""
    inputs = {}
    pieces = []

    # Categories: empty slot -> lookup table
    for name in categorical:
        input_dim, output_dim = categorical[name]
        inputs[name] = keras.Input(shape=(), dtype='int32', name=name)
        pieces.append(layers.Embedding(input_dim, output_dim, name=name + '_embedding')(inputs[name]))

    # Numbers: empty slot -> reshape (batch,) to (batch, 1)
    for name in numeric:
        inputs[name] = keras.Input(shape=(), dtype='float32', name=name)
        pieces.append(layers.Reshape((1,), name=name + '_reshape')(inputs[name]))

    # Glue -> mix -> dropout -> 64 numbers -> length 1
    x = layers.Concatenate(name=tower_name + '_concat')(pieces)
    x = layers.Dense(128, activation='relu', kernel_initializer='he_normal', name=tower_name + '_dense_1')(x)
    x = layers.Dropout(0.2, name=tower_name + '_dropout')(x)
    x = layers.Dense(64, name=tower_name + '_dense_out')(x)
    vector = layers.UnitNormalization(name=tower_name + '_normalise')(x)

    return keras.Model(inputs=inputs, outputs=vector, name=tower_name)


In [42]:
user_tower = build_tower(user_categorical, user_numeric, 'user_tower')
print('User tower parameters:', format(user_tower.count_params(), ','), '| expected 14,236,160 (same as Step 8)')


User tower parameters: 14,236,160 | expected 14,236,160 (same as Step 8)


In [43]:
# name: (vocab size + 1, embedding size)
item_categorical = {
    'article_idx':                      (len(article_vocab) + 1, 32),                              # 38,603   -> tens of thousands: 32
    'product_code_idx':                 (len(load_vocab('product_code')) + 1, 32),                 # 19,354   -> tens of thousands: 32
    'product_type_no_idx':              (len(product_type_vocab) + 1, 16),                         # 123      -> hundreds: 16
    'department_no_idx':                (len(department_vocab) + 1, 16),                           # 286      -> hundreds: 16
    'product_group_name_idx':           (len(load_vocab('product_group_name')) + 1, 8),            # 18       -> tens: 8
    'graphical_appearance_name_idx':    (len(load_vocab('graphical_appearance_name')) + 1, 8),     # 30
    'colour_group_name_idx':            (len(load_vocab('colour_group_name')) + 1, 8),             # 50
    'perceived_colour_master_name_idx': (len(load_vocab('perceived_colour_master_name')) + 1, 8),  # 20
    'section_name_idx':                 (len(load_vocab('section_name')) + 1, 8),                  # 56
    'garment_group_name_idx':           (len(load_vocab('garment_group_name')) + 1, 8),            # 21
    'index_group_name_idx':             (len(index_group_vocab) + 1, 4),                           # 5        -> tiny: 4
}
item_numeric = ['price_scaled']

for name in item_categorical:
    print(name, item_categorical[name])


article_idx

 (38604, 32)
product_code_idx (19355, 32)
product_type_no_idx (124, 16)
department_no_idx (287, 16)
product_group_name_idx (19, 8)
graphical_appearance_name_idx (31, 8)
colour_group_name_idx (51, 8)
perceived_colour_master_name_idx (21, 8)
section_name_idx (57, 8)
garment_group_name_idx (22, 8)
index_group_name_idx (6, 4)


In [44]:
item_tower = build_tower(item_categorical, item_numeric, 'item_tower')
item_tower.summary()
print('Item tower parameters:', format(item_tower.count_params(), ','), '| expected 1,890,352')


Model: "item_tower"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ article_idx         │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_code_idx    │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_type_no_idx │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ department_no_idx   │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_group_name… │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ graphical_appearan… │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ colour_group_name_… │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ perceived_colour_m… │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ section_name_idx    │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ garment_group_name… │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ index_group_name_i… │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ price_scaled        │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ article_idx_embedd… │ (None, 32)        │  1,235,328 │ article_idx[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_code_idx_e… │ (None, 32)        │    619,360 │ product_code_idx… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_type_no_id… │ (None, 16)        │      1,984 │ product_type_no_… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ department_no_idx_… │ (None, 16)        │      4,592 │ department_no_id… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_group_name… │ (None, 8)         │        152 │ product_group_na

 Total params: 1,890,352 (7.21 MB)

 Trainable params: 1,890,352 (7.21 MB)

 Non-trainable params: 0 (0.00 B)

Item tower parameters: 1,890,352 | expected 1,890,352


In [45]:
customer_embedding = 444432 * 32
article_embedding = 38604 * 32
user_total = user_tower.count_params()
item_total = item_tower.count_params()

print('User tower:', format(user_total, ','), '| customer embedding', format(customer_embedding, ','),
      '=', round(customer_embedding / user_total * 100, 1), '%')
print('Item tower:', format(item_total, ','), '| article embedding ', format(article_embedding, ','),
      '=', round(article_embedding / item_total * 100, 1), '%')
print('Both towers:', format(user_total + item_total, ','))


User tower: 14,236,160 | customer embedding 14,221,824 = 99.9 %
Item tower: 1,890,352 | article embedding  1,235,328 = 65.3 %
Both towers: 16,126,512


In [46]:
user_keys = list(user_categorical.keys()) + user_numeric     # 4 categories + FN, Active
item_keys = list(item_categorical.keys()) + item_numeric     # 11 categories + price_scaled

print('User keys:', len(user_keys), '| expected 6')
print('Item keys:', len(item_keys), '| expected 12')
print('Every key used once?', len(user_keys) + len(item_keys) == len(features), '| expected True')


User keys: 6 | expected 6
Item keys: 12 | expected 12
Every key used once? True | expected True


In [47]:
for batch in dataset.take(1):
    user_batch = {}
    for name in user_keys:
        user_batch[name] = batch[name]
    item_batch = {}
    for name in item_keys:
        item_batch[name] = batch[name]

user_vectors = user_tower(user_batch, training=False)
item_vectors = item_tower(item_batch, training=False)
print('User vectors:', user_vectors.shape, '| expected (1024, 64)')
print('Item vectors:', item_vectors.shape, '| expected (1024, 64)')


User vectors: (1024, 64) | expected (1024, 64)
Item vectors: (1024, 64) | expected (1024, 64)


In [48]:
user_lengths = np.linalg.norm(user_vectors.numpy(), axis=1)    # one length per row (customer)
item_lengths = np.linalg.norm(item_vectors.numpy(), axis=1)

print('User lengths: min', round(user_lengths.min(), 6), ' max', round(user_lengths.max(), 6), '| expected 1 and 1')
print('Item lengths: min', round(item_lengths.min(), 6), ' max', round(item_lengths.max(), 6), '| expected 1 and 1')
print('All within 1e-5 of 1?',
      np.all(np.abs(user_lengths - 1) < 1e-5) and np.all(np.abs(item_lengths - 1) < 1e-5), '| expected True')


User lengths: min 1.0  max 1.0 | expected 1 and 1
Item lengths: min 1.0  max 1.0 | expected 1 and 1
All within 1e-5 of 1? True | expected True


In [49]:
scores = user_vectors.numpy() @ item_vectors.numpy().T     # (1024, 64) @ (64, 1024) = (1024, 1024)
print('Score grid:', scores.shape, '| expected (1024, 1024)')
print('Smallest / largest score:', round(scores.min(), 3), '/', round(scores.max(), 3), '| expected between -1 and 1')

diagonal = np.diag(scores)        # customer i with the article customer i actually bought
print('Average score, bought (diagonal):', round(diagonal.mean(), 4))
print('Average score, everything else:  ', round((scores.sum() - diagonal.sum()) / (1024 * 1024 - 1024), 4))


Score grid: (1024, 1024) | expected (1024, 1024)
Smallest / largest score: -0.497 / 0.444 | expected between -1 and 1
Average score, bought (diagonal): -0.0169
Average score, everything else:   -0.016
